## Parte 1: Extracción y almacenamiento de datos 

**Justificación técnica:** 

Fuente de datos elegida: API de CoinGecko (plan gratuito)

Elegí la API de CoinGecko porque ofrece datos de criptomonedas en una API REST gratuita (plan Demo: 30 llamadas/min, 10.000/mes, sin tarjeta) que cubre los dos tipos de datos que pedía la consigna desde una misma fuente.

*Nota importante:* para ejecutar, crear un archivo .env en la carpeta del notebook con la variable COINGECKO_KEY. Se incluye .env.example como referencia.

**Endpoints utilizados:**

Usé dos endpoints de la misma API. El primero, /coins/markets, devuelve datos de mercado (precio, market cap, volumen) que se actualizan constantemente; es mi fuente de datos temporales. 

El segundo, /coins/{id}, devuelve metadatos descriptivos de cada moneda (nombre, algoritmo de hash, fecha de génesis, descripción, links); es mi fuente de datos estáticos. Ambos se enlazan por el campo id.

**Técnicas de extracción:**

Para los metadatos apliqué extracción full, porque son datos que prácticamente no cambian: no tiene sentido acumular versiones, así que en cada corrida sobrescribo (mode="overwrite"). Para los precios apliqué extracción incremental (mode="append"), porque el objetivo es ir acumulando una serie de "estados" del mercado a lo largo del tiempo sin perder las anteriores.

**Almacenamiento en Delta Lake:**

Organicé el data lake siguiendo la *arquitectura Medallion*, que separa los datos en capas de calidad creciente (bronze, silver, gold). Dentro de bronze seguí la estructura capa / sistema origen / entidad: una carpeta por el sistema origen (coingecko) y, dentro, una carpeta por cada entidad o endpoint (coins_markets y coins_metadata). Usé rutas relativas para que funcione igual en local y en Colab.

El dataset temporal lo particioné únicamente por fecha_extraccion. Descarté particionar también por hora porque, por el volumen de datos de este trabajo, generar una subcarpeta por hora sería innecesario y fragmentaría los datos de más; con la fecha alcanza. La columna hora_extraccion se conserva igualmente dentro de la tabla como dato informativo de la extracción. La librería deltalake crea los directorios automáticamente si no existen.

In [6]:
import pandas as pd
import requests as req

In [7]:
#!pip install deltalake
import deltalake

In [8]:
#!pip install python-dotenv

In [9]:
import os
from dotenv import load_dotenv

load_dotenv()   # lee el archivo .env
api_key = os.getenv("COINGECKO_KEY")

headers = {"x-cg-demo-api-key": api_key}

#### Pruebas de endpoints

In [10]:
url1 = "https://api.coingecko.com/api/v3/coins/markets" # según doc: all the supported coins with price, market cap, volume and market related data
url2 = "https://api.coingecko.com/api/v3/coins/bitcoin" # all the metadata (image, websites, socials, description, contract address, etc.) from the CoinGecko coin page based on a particular coin ID

In [11]:
parametros = { "vs_currency": "ars"}
respuesta1 = req.get(url1, headers=headers, params=parametros)

In [12]:
print("Código de estado:", respuesta1.status_code)
print("Datos que devolvió:", respuesta1.json())

Código de estado: 200
Datos que devolvió: [{'id': 'bitcoin', 'symbol': 'btc', 'name': 'Bitcoin', 'image': 'https://coin-images.coingecko.com/coins/images/1/large/bitcoin.png?1696501400', 'current_price': 91928993, 'market_cap': 1843296590338313, 'market_cap_rank': 1, 'fully_diluted_valuation': 1843299532090356, 'total_volume': 63147952796689, 'high_24h': 92405129, 'low_24h': 88684506, 'price_change_24h': 2328451, 'price_change_percentage_24h': 2.5987, 'market_cap_change_24h': 47054219155938, 'market_cap_change_percentage_24h': 2.61959, 'circulating_supply': 20051143.0, 'total_supply': 20051175.0, 'max_supply': 21000000.0, 'ath': 180895872, 'ath_change_percentage': -49.18127, 'ath_date': '2025-10-09T13:25:33.789Z', 'atl': 1478.98, 'atl_change_percentage': 6215614.09926, 'atl_date': '2015-01-14T00:00:00.000Z', 'roi': None, 'last_updated': '2026-07-02T18:38:12.123Z'}, {'id': 'ethereum', 'symbol': 'eth', 'name': 'Ethereum', 'image': 'https://coin-images.coingecko.com/coins/images/279/large

In [13]:
respuesta2 = req.get(url2, headers=headers)
datos = respuesta2.json()

In [14]:
print("Código de estado:", respuesta2.status_code)
print("Claves disponibles:", list(datos.keys()))

Código de estado: 200
Claves disponibles: ['id', 'symbol', 'name', 'web_slug', 'asset_platform_id', 'platforms', 'detail_platforms', 'block_time_in_minutes', 'hashing_algorithm', 'categories', 'preview_listing', 'public_notice', 'additional_notices', 'has_supply_breakdown', 'localization', 'description', 'links', 'image', 'country_origin', 'genesis_date', 'sentiment_votes_up_percentage', 'sentiment_votes_down_percentage', 'watchlist_portfolio_users', 'market_cap_rank', 'market_cap_rank_with_rehypothecated', 'market_data', 'community_data', 'developer_data', 'status_updates', 'last_updated', 'tickers']


#### Construcción de Dataframes

**Datos temporales**

In [ ]:
datos_mercado_temporal = respuesta1.json()

df_mercado_temporal = pd.DataFrame(datos_mercado_temporal)

In [16]:
print(df_mercado_temporal.shape)
df_mercado_temporal.head()

(100, 26)


,id,symbol,name,image,current_price,market_cap,market_cap_rank,fully_diluted_valuation,total_volume,high_24h,...,total_supply,max_supply,ath,ath_change_percentage,ath_date,atl,atl_change_percentage,atl_date,roi,last_updated
0,bitcoin,btc,Bitcoin,https://coin-images.coingecko.com/coins/images...,91928993.00,1843296590338313,1,1843299532090356,6.314795e+13,92405129.00,...,2.005118e+07,21000000.0,1.808959e+08,-49.18127,2025-10-09T13:25:33.789Z,1478.98000,6.215614e+06,2015-01-14T00:00:00.000Z,None,2026-07-02T18:38:12.123Z
1,ethereum,eth,Ethereum,https://coin-images.coingecko.com/coins/images...,2533031.00,305692207827811,2,305692207827811,1.992291e+13,2557187.00,...,1.206833e+08,NaN,6.915671e+06,-63.37259,2025-09-13T03:56:53.029Z,4.10000,6.171467e+07,2015-10-20T00:00:00.000Z,"{'times': 35.84233372160696, 'currency': 'btc'...",2026-07-02T18:38:12.593Z
2,tether,usdt,Tether,https://coin-images.coingecko.com/coins/images...,1487.58,274146650408721,3,282279187169822,9.673272e+13,1490.81,...,1.897568e+11,NaN,1.492990e+03,-0.36239,2025-10-24T21:01:00.718Z,5.00000,2.966072e+04,2015-03-02T00:00:00.000Z,None,2026-07-02T18:37:51.423Z
3,binancecoin,bnb,BNB,https://coin-images.coingecko.com/coins/images...,834711.00,112504843161968,4,112504843161968,1.252511e+12,842593.00,...,1.347826e+08,200000000.0,1.947443e+06,-57.13809,2025-10-13T08:41:24.131Z,0.69458,1.201749e+08,2017-10-19T00:00:00.000Z,None,2026-07-02T18:38:13.123Z
4,usd-coin,usdc,USDC,https://coin-images.coingecko.com/coins/images...,1488.70,109254500309421,5,109329509923246,2.047463e+13,1492.16,...,7.343945e+10,NaN,1.492160e+03,-0.23049,2026-07-02T14:55:54.046Z,35.10000,4.140967e+03,2018-11-05T00:00:00.000Z,None,2026-07-02T18:38:01.449Z


**Datos estáticos**

In [17]:
monedas = ["bitcoin", "ethereum", "usd-coin"]
filas = []

In [18]:
for moneda in monedas:
    url = f"https://api.coingecko.com/api/v3/coins/{moneda}"
    respuesta_meta = req.get(url, headers=headers)
    datos = respuesta_meta.json()

    fila = {
        "id": datos["id"],
        "symbol": datos["symbol"],
        "name": datos["name"],
        "hashing_algorithm": datos["hashing_algorithm"],
        "genesis_date": datos["genesis_date"],
        "country_origin": datos["country_origin"],
        "market_cap_rank": datos["market_cap_rank"],
        "categories": datos["categories"],
        "description_en": datos["description"]["en"],
        "homepage": datos["links"]["homepage"],
        "whitepaper": datos["links"]["whitepaper"],
    }

    filas.append(fila)

In [19]:
df_metadata = pd.DataFrame(filas)
df_metadata

,id,symbol,name,hashing_algorithm,genesis_date,country_origin,market_cap_rank,categories,description_en,homepage,whitepaper
0,bitcoin,btc,Bitcoin,SHA-256,2009-01-03,,1,"[Smart Contract Platform, Layer 1 (L1), FTX Ho...",Bitcoin is the world's first decentralized cry...,[http://www.bitcoin.org],https://bitcoin.org/bitcoin.pdf
1,ethereum,eth,Ethereum,Ethash,2015-07-30,,2,"[Smart Contract Platform, Layer 1 (L1), Ethere...","Ethereum is a global, open-source platform for...",[https://www.ethereum.org/],https://ethereum.org/whitepaper/
2,usd-coin,usdc,USDC,NaN,NaN,US,5,"[Stablecoins, USD Stablecoin, Solana Ecosystem...",USDC is a fully collateralized US dollar stabl...,[https://www.circle.com/en/usdc],https://www.circle.com/legal/mica-usdc-whitepaper


#### Extracción y guardado

In [20]:
from datetime import datetime

In [ ]:
ahora = datetime.now()

# se agrega fecha y hora al DataFrame temporal
df_mercado_temporal["fecha_extraccion"] = ahora.strftime("%Y-%m-%d")
df_mercado_temporal["hora_extraccion"] = ahora.strftime("%H:%M")

In [22]:
df_mercado_temporal[["id", "current_price", "fecha_extraccion", "hora_extraccion"]].head()

,id,current_price,fecha_extraccion,hora_extraccion
0,bitcoin,91928993.00,2026-07-02,15
1,ethereum,2533031.00,2026-07-02,15
2,tether,1487.58,2026-07-02,15
3,binancecoin,834711.00,2026-07-02,15
4,usd-coin,1488.70,2026-07-02,15


In [23]:
from deltalake import write_deltalake
from deltalake import DeltaTable

In [24]:
ruta_metadata = "datalake/bronze/coingecko/coins_metadata"

write_deltalake(
    ruta_metadata,
    df_metadata,
    mode="overwrite")

In [25]:
dt = DeltaTable(ruta_metadata)
dt.to_pandas()

,id,symbol,name,hashing_algorithm,genesis_date,country_origin,market_cap_rank,categories,description_en,homepage,whitepaper
0,bitcoin,btc,Bitcoin,SHA-256,2009-01-03,,1,"[Smart Contract Platform, Layer 1 (L1), FTX Ho...",Bitcoin is the world's first decentralized cry...,[http://www.bitcoin.org],https://bitcoin.org/bitcoin.pdf
1,ethereum,eth,Ethereum,Ethash,2015-07-30,,2,"[Smart Contract Platform, Layer 1 (L1), Ethere...","Ethereum is a global, open-source platform for...",[https://www.ethereum.org/],https://ethereum.org/whitepaper/
2,usd-coin,usdc,USDC,NaN,NaN,US,5,"[Stablecoins, USD Stablecoin, Solana Ecosystem...",USDC is a fully collateralized US dollar stabl...,[https://www.circle.com/en/usdc],https://www.circle.com/legal/mica-usdc-whitepaper


In [26]:
ruta_temporal = "datalake/bronze/coingecko/coins_markets"

write_deltalake(
    ruta_temporal,
    df_mercado_temporal,
    mode="append",
    partition_by=["fecha_extraccion"])

In [27]:
dt_temporal = DeltaTable(ruta_temporal)
print("Filas guardadas:", len(dt_temporal.to_pandas()))

Filas guardadas: 100
